In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np
from itertools import product

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from itertools import product, combinations
from scipy.optimize import minimize
import matplotlib.pyplot as plt


In [2]:
# Generate synthetic data
np.random.seed(42)

In [3]:
version = 'vpce1.1'
file_in=f'Ge77_rates_CNP_{version}.csv'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
   

# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=['Radius[cm]','Thickness[cm]','NPanels', 'Theta[deg]', 'Length[cm]']
x_labels_out = ['Radius [cm]','Thickness [cm]','NPanels', 'Angle [deg]', 'Length [cm]']

y_err_label_cnp = 'Ge-77_CNP_err'
y_label_sim = 'rGe77[nuc/(kg*yr)]'

# Set parameter boundaries
xmin=[0,0,0,0,0]
xmax=[265,20,360,90,150]

# Set parameter boundaries for aquisition function
xlow=[90,2,4,0,1]
xhigh=[250,15,360,90,150]

# Assign costs
low_fidelity_cost = 1.
high_fidelity_cost = 2000.
n_fidelities = 2
# Set a fixed point in space for drawings
x_fixed = [160, 2, 40, 45, 20]
# number of sigma for error band drawing on prediction
factor=1.

# Get LF noise from file
#with open(f'in/{file_in}') as f:
#    first_line = f.readline()
#LF_noise=np.round(float(first_line.split(' +')[0].split('= ')[1]),3)
LF_noise = 0.028
# Get HF and LF data samples from file

data=pd.read_csv(f'in/{file_in}')
#data=data[[f'Mode', x_labels[0], x_labels[1], x_labels[2], x_labels[3], x_labels[4],y_label_cnp,y_err_label_cnp,y_label_sim]]


In [4]:
x_lf, x_hf, y_lf, y_hf = ([],[],[],[])
row_h=data.index[data['Mode'] == 1]
row_l=data.index[data['Mode'] == 0]

x_hf = data.loc[data['Mode']==1.][x_labels].to_numpy()
y_hf = data.loc[data['Mode']==1.][y_label_sim].to_numpy()

x_lf = data.loc[data['Mode']==0.][x_labels].to_numpy()
y_lf = data.loc[data['Mode']==0.][ y_label_sim].to_numpy()


In [5]:
data=pd.read_csv("./in/hf_validation_data_v1.2.csv")
x_test = data.loc[data['Mode']==1.][x_labels].to_numpy()
y_true_test = data.loc[data['Mode']==1.][y_label_sim].to_numpy()

In [6]:
# Generate multivariate Legendre polynomial basis with interaction terms
def multivariate_legendre_with_interactions(order, x):
    """Generate multivariate Legendre polynomial basis with interactions."""
    n_samples, n_features = x.shape
    degrees = list(product(range(order + 1), repeat=n_features))
    basis = []
    for degree in degrees:
        term = np.ones(n_samples)
        for i, d in enumerate(degree):
            term *= np.polynomial.legendre.Legendre.basis(d)(x[:, i])
        basis.append(term)
    
    # Add interaction terms
    for i, j in combinations(range(n_features), 2):
        basis.append(x[:, i] * x[:, j])
    
    return np.vstack(basis).T

In [8]:
def cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5):
    errors = []
    kf = KFold(n_splits=5)
    for order in range(1, max_order + 1):
        phi_lf = multivariate_legendre_with_interactions(order, x_lf)
        phi_hf = multivariate_legendre_with_interactions(order, x_hf)

        # Fit LF model
        c_lf = np.linalg.lstsq(phi_lf, y_lf, rcond=None)[0]
        
        y_lf_hf_pred = phi_hf @ c_lf
        delta_hf = y_hf - y_lf_hf_pred

        mse_fold = []
        for train_idx, test_idx in kf.split(x_hf):
            x_train, x_test = x_hf[train_idx], x_hf[test_idx]
            y_train, y_test = y_hf[train_idx], y_hf[test_idx]

            phi_train = multivariate_legendre_with_interactions(order, x_train)
            phi_test = multivariate_legendre_with_interactions(order, x_test)
            
            # Bayesian fit for HF correction
            c_hf = np.linalg.lstsq(phi_train, y_train - phi_train @ c_lf, rcond=None)[0]
            y_pred_fold = phi_test @ c_hf + phi_test @ c_lf
            mse_fold.append(mean_squared_error(y_test, y_pred_fold))
        
        errors.append(np.mean(mse_fold))
    return np.argmin(errors) + 1

In [15]:
def bayesian_fit(phi, y, sigma_prior=1.0, sigma_noise=0.1, n_samples=1000):
    """Perform Bayesian inference to sample PCE coefficients."""
    n_coefficients = phi.shape[1]
    posterior_samples = []
    for i in range(n_samples):
        c = np.random.normal(0, sigma_prior, size=n_coefficients)
        likelihood = np.exp(-0.5 * np.sum((y - phi @ c) ** 2) / sigma_noise**2)
        prior = np.exp(-0.5 * np.sum(c**2) / sigma_prior**2)
        if np.random.rand() < likelihood * prior:
            posterior_samples.append(c)
        if i % 100 == 0:  # Debug every 100 iterations
            print(f"Iteration {i}: Likelihood={likelihood}, Prior={prior}, Accepted={len(posterior_samples)}")
    return np.array(posterior_samples)

In [10]:
def optimize_rho(phi_hf, y_hf, c_lf_samples, c_hf_samples):
    def loss_function(rho):
        y_hf_pred = rho * (phi_hf @ c_lf_samples.mean(axis=0)) + (phi_hf @ c_hf_samples.mean(axis=0))
        return mean_squared_error(y_hf, y_hf_pred)
    
    result = minimize(loss_function, x0=1.0, bounds=[(0.1, 10.0)])
    return result.x[0]

In [11]:
def predict_pce(x_test, phi_test, c_lf_samples, c_hf_samples, rho):
    y_pred_samples = []
    for c_lf, c_hf in zip(c_lf_samples, c_hf_samples):
        y_pred = rho * (phi_test @ c_lf) + (phi_test @ c_hf)
        y_pred_samples.append(y_pred)
    return np.mean(y_pred_samples, axis=0), np.std(y_pred_samples, axis=0)

In [18]:

# Cross-validate for optimal polynomial order
optimal_order = cross_validate_order(x_lf, y_lf, x_hf, y_hf, max_order=5)
print(f"Optimal Polynomial Order: {optimal_order}")
    
# Generate polynomial basis
phi_lf = multivariate_legendre_with_interactions(optimal_order, x_lf)
phi_hf = multivariate_legendre_with_interactions(optimal_order, x_hf)
def normalize_basis(phi):
    """Normalize columns of the basis matrix."""
    return phi / np.linalg.norm(phi, axis=0)

phi_lf = normalize_basis(phi_lf)
phi_hf = normalize_basis(phi_hf)

print("Phi LF shape:", phi_lf.shape)
print("Phi HF shape:", phi_hf.shape)
print("Example Phi LF row:", phi_lf[0])
print("Example Phi HF row:", phi_hf[0])
# Bayesian fit for LF and HF
print("Phi LF Shape:", phi_lf.shape)
print("Y LF Shape:", y_lf.shape)
print("Phi LF Example:", phi_lf[0])
print("Y LF Example:", y_lf[:5])
#c_lf_samples = bayesian_fit(phi_lf, y_lf, sigma_prior=1.0, sigma_noise=0.1, n_samples=1000)
c_lf_samples = bayesian_fit(phi_lf, y_lf, sigma_prior=5.0, sigma_noise=10., n_samples=1000)
if c_lf_samples.size == 0:
    print("Warning: No posterior samples generated for LF. Falling back to MLE.")
    c_lf_samples = np.array([np.linalg.lstsq(phi_lf, y_lf, rcond=None)[0]])
y_lf_hf_pred = phi_hf @ c_lf_samples.mean(axis=0)
delta_hf = y_hf - y_lf_hf_pred
c_hf_samples = bayesian_fit(phi_hf, delta_hf, sigma_prior=1.0, sigma_noise=0.01, n_samples=1000)


# Optimize scaling factor
rho_optimized = optimize_rho(phi_hf, y_hf, c_lf_samples, c_hf_samples)
print(f"Optimized rho: {rho_optimized}")

Optimal Polynomial Order: 1
Phi LF shape: (309, 42)
Phi HF shape: (10, 42)
Example Phi LF row: [0.05688801 0.07799699 0.02105116 0.06068547 0.00815447 0.02786993
 0.00269014 0.00952183 0.03514699 0.05913746 0.01380253 0.0512416
 0.00489665 0.02170381 0.00175179 0.00793936 0.06123675 0.08540326
 0.02289507 0.06606704 0.00857222 0.02850426 0.00286693 0.00975437
 0.03753145 0.06337988 0.0150599  0.05518607 0.00521661 0.02214887
 0.00190789 0.00816356 0.03753145 0.00857222 0.02289507 0.08540326
 0.00489665 0.01380253 0.05913746 0.00269014 0.02786993 0.06068547]
Example Phi HF row: [0.31622777 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.    

/var/folders/99/0svbmlns6xs9l9p55lcr912r0000gn/T/ipykernel_8591/2399687695.py:3: RuntimeWarning: Mean of empty slice.
  y_hf_pred = rho * (phi_hf @ c_lf_samples.mean(axis=0)) + (phi_hf @ c_hf_samples.mean(axis=0))
/Users/aschuetz/.local/modules/miniconda/miniconda3/envs/legend/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

In [ ]:
# Predict on test data
phi_test = multivariate_legendre_with_interactions(optimal_order, x_test)
y_pred_mean, y_pred_std = predict_pce(x_test, phi_test, c_lf_samples, c_hf_samples, rho_optimized)

# Plot predictions
plt.figure(figsize=(10, 6))
plt.plot(range(len(y_pred_mean)), y_pred_mean, label="Predicted HF Mean", color="green")
plt.fill_between(range(len(y_pred_mean)), y_pred_mean - 2 * y_pred_std, y_pred_mean + 2 * y_pred_std, alpha=0.3, label="Uncertainty")
plt.plot(range(len(y_true_test)), y_true_test, label="True HF", linestyle="--", color="blue")
plt.xlabel("Sample Index")
plt.ylabel("Output")
plt.title("Bayesian Multi-Fidelity PCE")
plt.legend()
plt.grid(True)
plt.show()